# JWST mosaics, PSF matching, and coadds

Streamlined notebook. Critical methods live in `mosaic.py` (`create_default_mosaic`,
`create_ccddata`, `update_photmjsr`, `coadd`) and `jwst123.py` (`generate_level3_mosaic`,
`jwst_phot`). Sky-region parsing uses `common.mast.parse_s_region`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import shapely
from astropy.io import fits
from astropy import wcs
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.convolution import convolve_fft
from astropy.stats import sigma_clipped_stats as scs
from photutils.psf.matching import resize_psf, create_matching_kernel, CosineBellWindow

from common.mast import parse_s_region, polygons_from_obs_table
from nbutils import input_list, create_filter_table
from mosaic import (
    create_default_mosaic,
    create_ccddata,
    update_photmjsr,
    coadd,
)
from jwst123 import generate_level3_mosaic, jwst_phot

## Level-3 mosaic from aligned CAL / JHAT frames


In [ ]:
workdir = Path('jwstred_temp_dolphot')
jhat_dir = workdir / 'jhat'
input_images = sorted(str(p) for p in jhat_dir.glob('*jhat.fits'))
print(len(input_images), 'input images')
table = input_list(input_images) if input_images else None
table

In [ ]:
# Build a single-filter or multi-filter Level-3 mosaic
# mosaic_name = generate_level3_mosaic(input_images, outdir=str(workdir / 'out'))
# mosaic_name

# Or use mosaic.create_default_mosaic for a filter subset:
# create_default_mosaic(input_images, outdir=str(workdir / 'mosaic'), filt='f200w')

## Sky footprint from S_REGION


In [ ]:
if table is not None and len(table):
    pgons = [parse_s_region(fits.open(im)['SCI'].header['S_REGION']) for im in table['image']]
    net_field = shapely.unary_union(pgons)
    net_field
else:
    net_field = None
    print('No table loaded; skip footprint cell')

## Coadd matched mosaics

`coadd` inverse-variance / duration-weights SCI frames and writes SCI/ERR/WHT HDUs.


In [ ]:
# Example after PSF-matching short-wavelength mosaics to a long-wavelength reference:
# ref_files = [
#     str(workdir / 'mosaic' / 'f200w_i2d.fits'),
#     str(workdir / 'mosaic' / 'ngc628_f150w_con_f200w.fits'),
# ]
# coadd(ref_files, 'F200W', filename=str(workdir / 'mosaic' / 'coadd_i2d.fits'))

## Photometry check on coadd


In [ ]:
# coadd_path = str(workdir / 'mosaic' / 'coadd_i2d.fits')
# refcat, photfile = jwst_phot(coadd_path)
# plt.hist(refcat['mag'][refcat['mag'] > 20], bins=40)
# plt.xlabel('mag'); plt.ylabel('N'); plt.show()
# refcat